# BASA JAWA Edge — Latih, tuning, telusuri, dan ekspor
Notebook utama untuk siswa SMA dengan pendamping. Gunakan CPU Colab. Dataset sekunder sudah dibundel; tidak perlu membuat korpus baru.

Tiga keluarga classifier: Logistic Regression, Linear SVM, dan Multinomial Naive Bayes. Lima pengaturan fitur dikombinasikan dengan tiga nilai parameter tiap classifier, sehingga ada **45 percobaan**. Model dipilih memakai validasi dan batas JSON 512 KiB yang ditetapkan sebelum test. PWA melakukan prediksi di perangkat pengguna.

**Berkas yang harus disimpan:** protokol, manifest lingkungan/kode, partisi, tabel tuning, keputusan pemilihan, prediksi test, trace fitur, model terpilih, dan rilis PWA. Hasil yang muncul adalah hasil runtime Anda. Jangan memakai angka contoh sebagai hasil Colab.


## 1 Hubungkan notebook dengan kode GitHub
Unggah isi paket awal ke repositori GitHub, lalu ubah REPO_URL. Notebook yang dibuka melalui GitHub tidak otomatis membawa seluruh berkas proyek ke runtime; sel ini melakukan clone. Untuk percobaan awal tanpa repositori, gunakan MODE="upload_zip" dan unggah ZIP paket.

Jalankan setiap sel berurutan. Bila ingin melihat kode fungsi, buka `src/lab.py` di panel Files atau GitHub. Nama fungsi mengikuti tahapan notebook.


In [ ]:
import os,sys,subprocess,re,json
from pathlib import Path
for name in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ[name]='1'
# Isi alamat repositori publik Anda setelah mengunggah paket awal ke GitHub.
REPO_URL = "https://github.com/USERNAME/basa-jawa-edge.git"
REPO_REF = "main"   # Untuk mengulang persis, ganti dengan commit SHA yang telah dicatat.
MODE = "github"     # Pilihan lain: "upload_zip" bila repositori belum dibuat.
if os.environ.get("BASA_REPO_ROOT"):
    ROOT = Path(os.environ["BASA_REPO_ROOT"]).resolve()  # pemeriksaan lokal oleh pengembang
elif MODE == "github":
    if not re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?",REPO_URL) or "USERNAME" in REPO_URL:
        raise ValueError("Ganti REPO_URL dengan repositori GitHub Anda, lalu jalankan sel ini lagi.")
    ROOT = Path("/content/basa-jawa-edge")
    if not ROOT.exists():subprocess.run(["git","clone",REPO_URL,str(ROOT)],check=True)
    subprocess.run(["git","-C",str(ROOT),"fetch","origin"],check=True)
    resolved_ref='origin/main' if REPO_REF=='main' else REPO_REF
    # Detached checkout mengunci sumber pada commit yang benar-benar diambil.
    # Git akan berhenti bila ada perubahan lokal yang bertabrakan; tidak ada reset otomatis.
    subprocess.run(["git","-C",str(ROOT),"checkout","--detach",resolved_ref],check=True)
else:
    from google.colab import files
    import zipfile
    uploads=files.upload()
    archive=next((name for name in uploads if name.endswith('.zip')),None)
    if archive is None:raise ValueError("Pilih ZIP paket repositori.")
    target=Path('/content/basa_upload');target.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            path=(target/member.filename).resolve()
            if not path.is_relative_to(target.resolve()):raise ValueError('Path ZIP tidak aman')
        z.extractall(target)
    matches=list(target.rglob('protocol/experiment.json'))
    if len(matches)!=1:raise ValueError('ZIP harus memuat satu folder proyek')
    ROOT=matches[0].parent.parent
assert (ROOT/'src/lab.py').exists(), ROOT
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
print('Folder proyek:',ROOT)


## 2 Siapkan pustaka sebelum mengimpor model
Sel ini hanya memasang pustaka utama bila versinya berbeda. Versi semua pustaka yang benar-benar digunakan akan dicatat ke manifest.


In [ ]:
import importlib.metadata
for name in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ[name]='1'  # Tetapkan sebelum NumPy/scikit-learn diimpor.
wanted={'scikit-learn':'1.8.0','joblib':'1.5.3','threadpoolctl':'3.6.0'}
needed=[]
for package,version in wanted.items():
    try:installed=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:installed=None
    if installed!=version:needed.append(f'{package}=={version}')
if needed:subprocess.run([sys.executable,'-m','pip','install','-q',*needed],check=True)
import numpy as np,pandas as pd,sklearn
try:
    from IPython.display import display,Image
except ImportError:
    def display(value):print(value)
    def Image(filename):return filename
from src import lab
if sklearn.__version__!='1.8.0':raise RuntimeError('Restart runtime, lalu jalankan lagi dari sel pertama.')
print('Python',sys.version.split()[0],'| scikit-learn',sklearn.__version__,'| NumPy',np.__version__)


## 3 Kunci protokol sebelum melihat hasil test
Baca parameter dan aturan pemilihan. Perubahan parameter dilakukan di `protocol/experiment.json`, disimpan sebagai commit, lalu dicatat pada run baru. Test tidak digunakan untuk memilih konfigurasi. Batas 512 KiB adalah batas desain paket JSON, bukan klaim batas memori smartphone.


In [ ]:
CONFIG=json.loads((ROOT/'protocol/experiment.json').read_text())
print(json.dumps(CONFIG,indent=2,ensure_ascii=False))
RUN,RUN_META=lab.start_run(ROOT,CONFIG,run_id=os.environ.get('BASA_RUN_ID'))
print('Run ID:',RUN_META['run_id'])
print('Commit:',RUN_META['source_git_commit'])
print('Hash kode:',RUN_META['training_code_sha256'])
print('Semua hasil disimpan di:',RUN)


## 4 Audit dataset dan makna label
Hash memeriksa bahwa berkas sama dengan snapshot yang ditetapkan. `translation.csv` menyediakan kalimat Jawa dan label, sedangkan `conversation.csv` menjadi materi dialog. Label bersumber dari peneliti asli. Tabel awal hanya audit, bukan hasil prediksi model.


In [ ]:
ROWS,DIALOGUES=lab.load_snapshot(ROOT,CONFIG,RUN)
DATA_HASH=lab.digest(ROOT/'data/translation.csv')
display(pd.DataFrame(ROWS).head(8))
display(pd.DataFrame([{'label':i,'nama':name,'baris':sum(int(r['label'])==i for r in ROWS)} for i,name in lab.LABELS.items()]))
print('Kalimat:',len(ROWS),'Dialog:',len(DIALOGUES),'Hash:',DATA_HASH)


## 5 Pisahkan keluarga kalimat sebelum pelatihan
`safe_groups` menyatukan group dan kalimat paralel yang identik setelah normalisasi. Model tidak boleh belajar satu versi kalimat lalu diuji pada versi tingkat tutur lain dari keluarga yang sama. Semua perhitungan kosakata dan IDF hanya memakai train. `partitions.csv` menyimpan identitas baris sehingga pembagian bisa dilacak.


In [ ]:
PARTS,GROUPS=lab.partition(ROWS,CONFIG,RUN)
display(pd.DataFrame([{'partisi':name,'baris':len(ids),'komponen_grup':len(set(GROUPS[ids]))} for name,ids in PARTS.items()]))
print('Irisan grup kosong: diperiksa dengan assert di lab.partition.')
# Buka fungsi jika ingin menelusuri proses pembagian:
import inspect
print(inspect.getsource(lab.partition))


## 6 Definisikan percobaan dan baseline
Tabel berikut adalah seluruh rencana percobaan. `C` mengatur regularisasi LR/SVM; `alpha` mengatur smoothing Naive Bayes. Mulai dari baseline kata + LR dengan C=4 agar alur fit dan validasi terlihat sebelum tuning.


In [ ]:
SPECS=lab.trial_specs(CONFIG)
display(pd.DataFrame([{'trial':s['trial_id'],'fitur':s['feature']['name'],'keluarga':s['family'],s['parameter_name']:s['parameter_value']} for s in SPECS]))
BASELINE=next(s for s in SPECS if s['family']=='logistic_regression' and s['feature']['name']=='word_full' and s['parameter_value']==4)
BASE_MODEL,BASE_RECORD=lab.fit_trial(BASELINE,ROWS,PARTS,CONFIG,RUN_META,DATA_HASH)
print('Baseline:',BASE_RECORD)
print(BASE_MODEL)
print(inspect.getsource(lab.make_pipeline))
print(inspect.getsource(lab.fit_trial))


## 7 Jalankan tuning dan simpan setiap percobaan
Baca loop ini: buat pipeline, fit train, prediksi validation, hitung macro-F1, ukur JSON, lalu tulis hasil. Tidak ada prediksi test di fungsi `fit_trial`. Angka durasi berasal dari runtime Colab dan tidak menggambarkan kecepatan smartphone.


In [ ]:
MODELS={BASELINE['trial_id']:BASE_MODEL};RECORDS=[]
for spec in SPECS:
    if spec['trial_id']==BASELINE['trial_id']:model,record=BASE_MODEL,BASE_RECORD
    else:model,record=lab.fit_trial(spec,ROWS,PARTS,CONFIG,RUN_META,DATA_HASH)
    MODELS[spec['trial_id']]=model;RECORDS.append(record)
    lab.write_csv(RUN/'trials.csv',RECORDS)
    print(record['trial_id'],record['family'],record['feature'],
          f"val F1={record['validation_macro_f1']:.4f}",
          f"JSON={record['model_json_bytes']/1024:.1f} KiB",'eligible=',record['eligible'])
TABLE=pd.DataFrame(RECORDS)
display(TABLE.sort_values('validation_macro_f1',ascending=False).head(12))
lab.plot_results(RECORDS,RUN)
display(Image(filename=str(RUN/'validation_tradeoff.png')))


## 8 Pilih model dengan aturan yang sudah dikunci
Model harus di bawah batas JSON. Dari kandidat yang macro-F1 validasinya dekat dengan kandidat eligible terbaik (selisih maksimal 0,01), pilih JSON terkecil. Ini memilih keseimbangan yang ditetapkan; hasilnya dapat berbeda dari model dengan skor absolut tertinggi. Model tetap dilatih pada train, tanpa refit setelah test.


In [ ]:
SELECTED=lab.select_trial(RECORDS,CONFIG,RUN)
CHAMPION=MODELS[SELECTED['trial_id']]
SELECTED_SPEC=next(s for s in SPECS if s['trial_id']==SELECTED['trial_id'])
print(json.dumps(json.loads((RUN/'selection.json').read_text()),indent=2))


## 9 Evaluasi test satu kali setelah keputusan final
Jangan mengubah parameter berdasarkan tabel ini. Fungsi menolak evaluasi ulang pada run yang sama agar urutan kerja terlihat. Jika tujuan atau aturan penelitian diubah setelah melihat test, butuhkan evaluasi baru yang belum dipakai untuk memilih model.


In [ ]:
TEST=lab.evaluate_test_once(CHAMPION,SELECTED,ROWS,PARTS,RUN)
print('Test macro-F1:',TEST['macro_f1'],'Akurasi:',TEST['accuracy'])
display(pd.DataFrame(TEST['confusion_matrix'],index=lab.LABELS.values(),columns=lab.LABELS.values()))
display(pd.DataFrame(TEST['per_class']).T)


## 10 Telusuri satu prediksi hingga kontribusi fitur
Untuk setiap kelas, skor adalah intersep ditambah jumlah kontribusi fitur. CSV menampilkan n-gram, frekuensi, IDF, TF-IDF, dan kontribusi per kelas. SVM mengeluarkan decision score; softmax skor SVM bukan probabilitas yang terkalibrasi. Antarmuka PWA hanya menampilkan perkiraan label.


In [ ]:
EXPORTED=lab.portable_model(CHAMPION,SELECTED_SPEC,RUN_META,DATA_HASH)
TRACE_ROWS,TRACE=lab.trace_sentence(CHAMPION,EXPORTED,'Kula nedha pecel dinten puniki.',RUN)
display(pd.DataFrame(TRACE_ROWS).head(20));print(json.dumps(TRACE,indent=2,ensure_ascii=False))


## 11 Ekspor bobot untuk browser dan ikat identitas rilis
PWA menerima JSON berisi bobot hasil pelatihan ini. Ekspor tidak melatih ulang. JavaScript mendukung ketiga classifier linear melalui penjumlahan bobot fitur. Kartu contoh dan kuis tetap bersumber dari dataset. Simpan Run ID dan versi rilis untuk mencocokkan aplikasi dengan eksperimen.


In [ ]:
EXPORT=lab.export_pwa(ROOT,RUN,CHAMPION,SELECTED_SPEC,RUN_META,ROWS,DIALOGUES,PARTS)
print(json.dumps(EXPORT,indent=2))
print('Model browser:',ROOT/'docs/assets/model.json')
print('Pemeriksaan JavaScript dilakukan di halaman test.html pada browser sasaran.')


## 12 Unduh hasil sebelum runtime berakhir
ZIP pembaruan memuat seluruh folder docs/ dan bukti eksperimen kecil untuk GitHub. Model Python disimpan terpisah di run; checkpoint besar tidak diperlukan untuk PWA. Simpan juga notebook melalui File → Save a copy in GitHub atau unduh .ipynb dengan output. File runtime tidak otomatis tersimpan bersama notebook.


In [ ]:
UPDATE_ZIP=lab.make_update_zip(ROOT,RUN)
print('ZIP pembaruan:',UPDATE_ZIP)
try:
    from google.colab import files
    files.download(str(UPDATE_ZIP))
except ImportError:
    print('Pemeriksaan lokal: unduh arsip ini secara manual.')


## 13 Publikasikan dan uji PWA
Ekstrak ZIP pembaruan pada salinan repositori di Windows. Commit folder docs/ dan bukti run, lalu push melalui GitHub Desktop. Aktifkan Pages dari main dan /docs. Buka HTTPS, tunggu Siap offline, cocokkan Run ID/Versi dengan pwa_export.json, lalu buka test.html. Setelah itu lakukan pemeriksaan offline dan uji empat tugas kepada siswa.

Fine-tuning DistilBERT tersedia di notebook 02 sebagai eksperimen opsional. Checkpoint tersebut tidak otomatis dijalankan oleh PWA ringan.

Sumber platform: https://research.google.com/colaboratory/faq.html ; https://docs.github.com/en/pages/getting-started-with-github-pages/configuring-a-publishing-source-for-your-github-pages-site .
